# 07. Optional: the same RAG in LangChain with LangSmith tracing

**Goal.** Show how the hand-built pipeline maps onto LangChain (orchestration) and LangSmith (observability).

**Playbook steps covered:** 3 (architecture alternatives), 9 (observability)

> Every step below has three parts: a **rationale** (why we do it), the **code** with line-level comments
> (how we do it), and an **evaluation** (how we know it worked). All notebooks run **offline** by default
> (hashing embedder + extractive baseline LLM). Switch to Claude and sentence-transformers with
> `AD_RAG_LLM_BACKEND=anthropic` and `AD_RAG_EMBEDDING_BACKEND=sentence-transformers`.

> The corpus is **synthetic educational content**. Nothing here is clinical guidance.

In [1]:
# ---- Setup: make the notebook runnable from anywhere (GitHub clone, Colab, Docker) ----
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Find the repo root by walking up until we see configs/config.yaml.
ROOT = Path.cwd()
while not (ROOT / "configs" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.environ["AD_RAG_HOME"] = str(ROOT)          # tell the package where the repo lives

# Fresh environment (e.g. Colab)? Install the package in editable mode first:
#   %pip install -q -e ".[notebooks]"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ad_rag import load_settings, build_pipeline

settings = load_settings()                     # typed config from configs/config.yaml (+ env overrides)
print("repo root :", ROOT)
print("embedder  :", settings.embedding.backend, "| llm backend:", settings.llm.backend)

repo root : /home/yodeyemi/alzheimers-llm-rag-agent
embedder  : hashing | llm backend: extractive


## Component mapping

| This repo | LangChain / LangSmith |
|---|---|
| `chunking.py` | `RecursiveCharacterTextSplitter` |
| `embeddings.py` | `Embeddings` subclass (we wrap our own) |
| `vectorstore.py` | `FAISS` / `InMemoryVectorStore` / Chroma / pgvector |
| `rag.build_prompt` | `ChatPromptTemplate` |
| `llm.py` | `ChatAnthropic` (or any chat model) |
| `RAGPipeline.ask` | LCEL chain with the `|` operator |
| `monitoring.py`, audit | **LangSmith tracing** (sidecar, not in the request path) |
| `evaluation.py` | LangSmith datasets and `evaluate()` |

LangSmith is **not** in the request path. LangChain emits a trace per chain run (each retrieval, prompt, model call, with inputs,
outputs, latency, tokens). You get trace trees, dashboards, datasets and online evaluators for free. The trade-off versus this
repo's own audit log: LangSmith is a hosted service, so check data-residency and PHI policy before sending any clinical text
to it (our PHI gate and redaction still apply, and self-hosted LangSmith is available).

### Step 1: Build the LCEL chain over our existing index

**Rationale.** Reusing the same embedder and chunks keeps results comparable with the hand-rolled pipeline and proves the design is portable: swapping the orchestration layer does not change the data layer.

**How this step is evaluated.** The chain answers with valid citations and agrees with the hand-rolled pipeline on the top source.

In [2]:
from ad_rag.langchain_chain import build_langchain_rag
from ad_rag.llm import ExtractiveLLM
from ad_rag.guardrails import extract_citations

pipe = build_pipeline(settings)
# Offline: wrap our extractive LLM as a Runnable. Online: pass chat_model=ChatAnthropic(model=settings.llm.model).
chain = build_langchain_rag(pipe.store, pipe.embedder, llm=ExtractiveLLM(), k=4)

q = "Why do assays use two cutoffs instead of one?"
lc_answer = chain.invoke(q)                       # the whole RAG flow is one .invoke()
print(lc_answer)
print("\nhand-rolled top source:", pipe.ask(q).sources[0]["source"], "| langchain citations:", sorted(set(extract_citations(lc_answer))))
assert "two_cutoff_interpretation.md" in extract_citations(lc_answer)

- Many assays report results using two cutoffs, which creates a low band, an intermediate band and a high band. [source: two_cutoff_interpretation.md]
- Using two cutoffs reduces misclassification at the cost of a share of patients needing further testing. [source: two_cutoff_interpretation.md]
- Results between the two cutoffs fall in an intermediate zone where accuracy is lower. [source: two_cutoff_interpretation.md]

hand-rolled top source: two_cutoff_interpretation.md | langchain citations: ['two_cutoff_interpretation.md']


### Step 2: Inspect the chain structure

**Rationale.** LCEL chains are inspectable objects. Printing the graph shows the same stages as our own diagram: parallel retrieval and passthrough, prompt, model, parser.

**How this step is evaluated.** Visual confirmation that the stages match the architecture.

In [3]:
try:
    chain.get_graph().print_ascii()
except Exception as e:                           # print_ascii needs an optional dependency (grandalf)
    print("(install grandalf for ASCII graphs)", type(e).__name__)
    print([type(s).__name__ for s in chain.steps] if hasattr(chain, "steps") else chain)

(install grandalf for ASCII graphs) ImportError
['RunnableParallel', 'ChatPromptTemplate', 'RunnableLambda', 'StrOutputParser']


### Step 3: Turn on LangSmith tracing (no code changes)

**Rationale.** Tracing is enabled purely through environment variables. Every `chain.invoke` then appears in the LangSmith UI as a trace tree with latency and token usage per step, where you can attach user feedback, build datasets from real traces and run LLM-as-judge evaluators.

**How this step is evaluated.** In the LangSmith UI: a trace per request, p95 latency chart, error rate, and feedback scores. Set alerts on those.

In [4]:
LANGSMITH_ENABLED = bool(os.environ.get("LANGCHAIN_API_KEY"))
if LANGSMITH_ENABLED:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"                 # the switch
    os.environ["LANGCHAIN_PROJECT"] = "ad-clinical-evidence-assistant"   # groups traces per application
    chain.invoke("How does chronic kidney disease affect plasma biomarkers?")
    print("Trace sent. Open https://smith.langchain.com and look at project 'ad-clinical-evidence-assistant'.")
else:
    print("LANGCHAIN_API_KEY not set: tracing disabled. To enable:\n"
          "  export LANGCHAIN_API_KEY=lsv2_...\n  export LANGCHAIN_TRACING_V2=true\n  export LANGCHAIN_PROJECT=ad-clinical-evidence-assistant")

LANGCHAIN_API_KEY not set: tracing disabled. To enable:
  export LANGCHAIN_API_KEY=lsv2_...
  export LANGCHAIN_TRACING_V2=true
  export LANGCHAIN_PROJECT=ad-clinical-evidence-assistant


## LangSmith evaluation (reference snippet, needs a LangSmith account)

```python
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()
ds = client.create_dataset("ad-golden-set")
client.create_examples(
    inputs=[{"question": g["question"]} for g in golden if g["category"] == "in_scope"],
    outputs=[{"must_contain": g["expected_keywords"]} for g in golden if g["category"] == "in_scope"],
    dataset_id=ds.id)

def contains_keywords(run, example):
    ok = all(k.lower() in run.outputs["output"].lower() for k in example.outputs["must_contain"])
    return {"key": "keyword_recall", "score": int(ok)}

evaluate(lambda x: {"output": chain.invoke(x["question"])}, data="ad-golden-set", evaluators=[contains_keywords])
```

## Which should you use?

* **Own pipeline (this repo's default):** fewest dependencies, full control over guardrails, audit and PHI handling, easy to validate.
* **LangChain + LangSmith:** faster to extend (many loaders, vector stores, agents via LangGraph) and richer tracing UI, at the cost of
  more dependencies and an external service in the loop. A good choice once the controls in `guardrails.py` and `monitoring.py` are
  ported as chain steps and callbacks.